In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time


In [2]:
categories = {
    "Travel": "https://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "Mystery": "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
    "Science Fiction": "https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html",
}

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CapstoneScraper/1.0)"}

book_data = []

for category_name, start_url in categories.items():
    page_url = start_url
    category_count = 0

    while page_url:
        response = requests.get(page_url, headers=HEADERS, timeout=20)
        response.encoding = "utf-8"
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        books = soup.find_all("article", class_="product_pod")

        for book in books:
            title_tag = book.h3.a
            price_tag = book.find("p", class_="price_color")
            rating_tag = book.find("p", class_="star-rating")
            availability_tag = book.find("p", class_="instock availability")

            # Skip only a genuinely incomplete listing so the final dataset
            # contains no null values in the required source columns.
            if not all([title_tag, price_tag, rating_tag, availability_tag]):
                continue

            title = title_tag.get("title", "").strip()
            price = price_tag.get_text(strip=True)
            rating_classes = rating_tag.get("class", [])
            rating = rating_classes[1] if len(rating_classes) > 1 else ""
            availability = availability_tag.get_text(" ", strip=True)

            if not all([title, price, rating, availability, category_name]):
                continue

            book_data.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": category_name,
            })
            category_count += 1

        # Continue until this category has no Next page.
        next_page = soup.find("li", class_="next")
        if next_page and next_page.a:
            page_url = urljoin(page_url, next_page.a["href"])
            time.sleep(0.3)
        else:
            page_url = None

    print(f"{category_name}: {category_count} books")

df = pd.DataFrame(book_data)

# Required project checks
required_columns = ["title", "price", "star_rating", "availability", "category"]
assert len(categories) == 4, "The scraper must use exactly 4 categories."
assert df["category"].nunique() == 4, "All 4 categories must appear in the dataset."
assert len(df) >= 70, f"Need at least 70 books, but only {len(df)} were scraped."
assert not df[required_columns].isna().any().any(), "Required columns contain null values."

print(f"\nTotal books in DataFrame: {len(df)}")
print(df["category"].value_counts())
df.head(10)

Travel: 11 books
Mystery: 32 books
Historical Fiction: 26 books
Science Fiction: 16 books

Total books in DataFrame: 85
category
Mystery               32
Historical Fiction    26
Science Fiction       16
Travel                11
Name: count, dtype: int64


,title,price,star_rating,availability,category
0,It's Only the Himalayas,£45.17,Two,In stock,Travel
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,£37.33,Three,In stock,Travel
5,A Summer In Europe,£44.34,Two,In stock,Travel
6,The Great Railway Bazaar,£30.54,One,In stock,Travel
7,A Year in Provence (Provence #1),£56.88,Four,In stock,Travel
8,The Road to Little Dribbling: Adventures of an...,£23.21,One,In stock,Travel
9,Neither Here nor There: Travels in Europe,£38.95,Three,In stock,Travel


In [3]:
df.shape

(85, 5)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         85 non-null     object
 1   price         85 non-null     object
 2   star_rating   85 non-null     object
 3   availability  85 non-null     object
 4   category      85 non-null     object
dtypes: object(5)
memory usage: 3.4+ KB


In [5]:
df.sample(2)

,title,price,star_rating,availability,category
71,"William Shakespeare's Star Wars: Verily, A New...",£43.30,Four,In stock,Science Fiction
19,A Time of Torment (Charlie Parker #14),£48.35,Five,In stock,Mystery


Stripping the currency symbol from price and converting it to a float column price_gbp.

In [6]:
# cleaning price column
# we generally use
# Remove the currency symbol from the 'price' column and store in 'price_gbp'
# df['price_gbp'] = df['price'].str.replace('Â£', '', regex=False)
# Convert the 'price_gbp' column (which now has no currency symbol) to numeric
#df['price_gbp'] = pd.to_numeric(df['price_gbp'], errors='coerce')

# But due to more effiecieny we are using regex here

# Use regex to strip any non-numeric characters except decimal point
df['price_gbp'] = pd.to_numeric(df['price'].str.replace(r"[^\d.]", "", regex=True), errors='coerce')

# Audit imputation
null_count = df['price_gbp'].isna().sum()
price_median = df['price_gbp'].median()
df['price_gbp'] = df['price_gbp'].fillna(price_median)
print(f"Imputed {null_count} missing values for price_gbp using median ({price_median:.2f})")

# Drop the original price column
df.drop(columns=['price'], inplace=True)

Imputed 0 missing values for price_gbp using median (33.26)


Converting the text star rating (One…Five) into an integer column rating (1–5).

In [7]:
# converting star rating column

rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['rating'] = df['star_rating'].map(rating_map)

# Audit imputation
null_count = df['rating'].isna().sum()
rating_median = df['rating'].median()
df['rating'] = df['rating'].fillna(rating_median).astype(int)
print(f"Imputed {null_count} missing values for rating using median ({rating_median})")

# Drop the original star_rating column
df.drop(columns=['star_rating'], inplace=True)

Imputed 0 missing values for rating using median (3.0)


Parsing the availability text into a boolean column in_stock.

In [8]:
# Convert "In stock" to True, everything else to False
df['in_stock'] = df['availability'].str.contains('In stock', case=False, na=False)

# Drop the original availability column
df.drop(columns=['availability'], inplace=True)

In [9]:
# Median imputation is now handled within the cleaning cells above.

In [10]:
#converting gbp to INR

df['price_inr'] = df['price_gbp'] * 105.50


In [11]:
# Generate category IDs dynamically to prevent mapping drift
unique_cats = df['category'].unique()
cat_mapping = {cat: i + 1 for i, cat in enumerate(unique_cats)}
df['category_id'] = df['category'].map(cat_mapping)
df.drop(columns=['category'], inplace=True)
print(f"Dynamic Category Mapping: {cat_mapping}")

Dynamic Category Mapping: {'Travel': 1, 'Mystery': 2, 'Historical Fiction': 3, 'Science Fiction': 4}


In [12]:
# Re arranging columns for better reading purpose
# Maintain only the final cleaned columns (using 'rating' and 'in_stock' instead of the dropped original columns)
df = df[['title','category_id','price_gbp','price_inr','rating','in_stock']]

Creating DataBase

In [13]:
import sqlite3
import os

# Run this before starting the database from scratch
if os.path.exists('booksdatabase.db'):
    os.remove('booksdatabase.db')

# Creating Database
conn = sqlite3.connect('booksdatabase.db')
cursor = conn.cursor()

# Enable foreign key support in SQLite
cursor.execute('PRAGMA foreign_keys = ON;')

# Use the dynamic mapping generated in the cleaning section
categories_data = [(cid, name) for name, cid in cat_mapping.items()]

# Create categories table
cursor.execute('''
    CREATE TABLE categories(
        category_id INTEGER PRIMARY KEY,
        category_name TEXT UNIQUE
    );
''')

# Insert categories
cursor.executemany('INSERT INTO categories(category_id, category_name) VALUES (?, ?)', categories_data)

# Create books table with proper normalized schema
cursor.execute('''
    CREATE TABLE books(
        book_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT,
        price_gbp REAL,
        price_inr REAL,
        rating INTEGER,
        in_stock INTEGER,
        category_id INTEGER,
        FOREIGN KEY (category_id) REFERENCES categories(category_id)
    );
''')
conn.commit()

In [14]:
# To preserve the schema (PK/FK), we avoid df.to_sql(if_exists='replace')
# Instead, we insert the data manually using executemany

# Convert boolean in_stock to 0/1 for SQLite
data_to_insert = df[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']].values.tolist()
# cast in_stock to int
for row in data_to_insert:
    row[4] = 1 if row[4] else 0

cursor.executemany('''
    INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
    VALUES (?, ?, ?, ?, ?, ?)
''', data_to_insert)

conn.commit()

sql queries

In [15]:
# Query using JOIN to verify data loading
query = '''SELECT b.*, c.category_name
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           '''
pd.read_sql(query, conn)

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,1,It's Only the Himalayas,45.17,4765.435,2,1,1,Travel
1,2,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,5214.865,4,1,1,Travel
2,3,See America: A Celebration of Our National Par...,48.87,5155.785,3,1,1,Travel
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,1,1,Travel
4,5,Under the Tuscan Sun,37.33,3938.315,3,1,1,Travel
...,...,...,...,...,...,...,...,...
80,81,Dune (Dune #1),54.86,5787.730,1,1,4,Science Fiction
81,82,Do Androids Dream of Electric Sheep? (Blade Ru...,51.48,5431.140,1,1,4,Science Fiction
82,83,Three Wishes (River of Time: California #1),44.18,4660.990,2,1,4,Science Fiction
83,84,The Last Girl (The Dominion Trilogy #1),36.26,3825.430,2,1,4,Science Fiction


In [ ]:
# 1. ORDER BY : all books and their categories with 5 star rating
query = '''SELECT b.title, c.category_name, b.rating
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           WHERE b.rating = 5
           ORDER BY b.title ASC
           LIMIT 10'''
pd.read_sql(query, conn)

In [ ]:
# 2. ORDER BY : all least rated books with their Categories

query = '''SELECT b.title, c.category_name, b.rating
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           WHERE b.rating = 1
           ORDER BY b.title ASC
           '''
pd.read_sql(query, conn)

In [ ]:
# 3. ORDER BY and LIMIT : Top 10 most expensive books
query = '''SELECT b.title, c.category_name, b.rating, b.price_inr
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           ORDER BY b.price_inr DESC
           LIMIT 10'''
pd.read_sql(query, conn)

In [19]:
# 4. DISTINCT: List all unique ratings present in the dataset
query = '''SELECT DISTINCT rating FROM books ORDER BY rating DESC'''
pd.read_sql(query, conn)

,rating
0,5
1,4
2,3
3,2
4,1


In [20]:
# 5. BETWEEN: Books in a specific price range
query = '''SELECT b.title, c.category_name, b.rating, b.price_inr
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           WHERE b.price_inr BETWEEN 1000 AND 5000
           ORDER BY b.rating DESC
           LIMIT 10'''
pd.read_sql(query, conn)

,title,category_name,rating,price_inr
0,"1,000 Places to See Before You Die",Travel,5,2751.440
1,What Happened on Beale Street (Secrets of the ...,Mystery,5,2676.535
2,The Silkworm (Cormoran Strike #2),Mystery,5,2431.775
3,The Girl You Lost,Mystery,5,1296.595
4,Mrs. Houdini,Historical Fiction,5,3191.375
5,The Passion of Dolssa,Historical Fiction,5,2987.760
6,Voyager (Outlander #3),Historical Fiction,5,2222.885
7,The Red Tent,Historical Fiction,5,3762.130
8,Between Shades of Gray,Historical Fiction,5,2193.345
9,While You Were Mine,Historical Fiction,5,4359.260


In [21]:
# 6. IN: Books in specific categories
query = '''SELECT b.title, c.category_name, b.rating, b.price_inr
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           WHERE c.category_name IN ('Travel', 'Mystery')
           ORDER BY b.rating ASC
           LIMIT 10'''
pd.read_sql(query, conn)

,title,category_name,rating,price_inr
0,"In a Dark, Dark Wood",Mystery,1,2070.965
1,A Murder in Time,Mystery,1,1755.520
2,That Darkness (Gardiner and Renner #1),Mystery,1,1468.560
3,Tastes Like Fear (DI Marnie Rome #3),Mystery,1,1127.795
4,Hide Away (Eve Duncan #20),Mystery,1,1249.120
5,The Cuckoo's Calling (Cormoran Strike #1),Mystery,1,2026.655
6,1st to Die (Women's Murder Club #1),Mystery,1,5694.890
7,The Great Railway Bazaar,Travel,1,3221.970
8,The Road to Little Dribbling: Adventures of an...,Travel,1,2448.655
9,The Last Mile (Amos Decker #2),Mystery,2,5719.155


In [22]:
# 7. DISTINCT: List all unique categories present in the category table
query = '''SELECT DISTINCT category_name FROM categories
           ORDER BY category_name ASC'''
pd.read_sql(query, conn)

,category_name
0,Historical Fiction
1,Mystery
2,Science Fiction
3,Travel


pd.merge

In [ ]:
# Reproducing the join using pd.merge (In-Memory Only)
# Build category_df from cat_mapping for a single source of truth
category_df = pd.DataFrame(list(cat_mapping.items()), columns=['category_name','category_id'])

# Use the original cleaned DataFrame 'df' for books
book_df_mem = df.copy()

merged_df = pd.merge(book_df_mem, category_df, on='category_id')
merged_df.head()

In [ ]:
# Reproducing the join query result with SQL for comparison

# A : all 5 star rated books with their Categories


query = '''SELECT b.title, c.category_name, b.rating
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           WHERE b.rating = 5
           ORDER BY b.title ASC
           '''
sql_result_top_rated = pd.read_sql(query, conn)
sql_result_top_rated

In [ ]:
# A : all 5 star rated books (Pandas merge vs SQL join)
# Use merged_df's own rating column to avoid index alignment issues
pandas_result_top_rated = merged_df[merged_df['rating'] == 5].sort_values(by='title', ascending=True)[['title', 'category_name', 'rating']]
pandas_result_top_rated

In [ ]:
# Final check: verify SQL and Pandas results match
# Display side-by-side for visual confirmation
comparison_top = pd.concat([sql_result_top_rated.reset_index(drop=True), pandas_result_top_rated.reset_index(drop=True)], axis=1, keys=["SQL", "Pandas"])
print("Side-by-Side Comparison (Top Rated):")
display(comparison_top)

pd.testing.assert_frame_equal(sql_result_top_rated.reset_index(drop=True), pandas_result_top_rated.reset_index(drop=True), check_dtype=False)
print("\nSQL and Pandas results match perfectly!")

In [ ]:
# Reproducing the join query result with SQL for comparison

# B : all least rated books with their Categories

query = '''SELECT b.title, c.category_name, b.rating
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           WHERE b.rating = 1
           ORDER BY b.title ASC
           '''
sql_result_least_rated = pd.read_sql(query, conn)
sql_result_least_rated

In [ ]:
# B :  Compare all least rated books (Pandas merge vs SQL join)
# Use merged_df's own rating column
pandas_result_least_rated = merged_df[merged_df['rating'] == 1].sort_values(by='title', ascending=True)[['title', 'category_name', 'rating']]
pandas_result_least_rated

In [29]:
# Final check: verify SQL and Pandas results match
pd.testing.assert_frame_equal(sql_result_least_rated.reset_index(drop=True), pandas_result_least_rated.reset_index(drop=True), check_dtype=False)
print("SQL and Pandas results match perfectly!")

SQL and Pandas results match perfectly!


In [ ]:
#  Top 10 most expensive books
query = '''SELECT b.title, c.category_name, b.rating, b.price_inr
           FROM books b
           INNER JOIN categories c ON b.category_id = c.category_id
           ORDER BY b.price_inr DESC, b.title ASC
           LIMIT 10'''
sql_result_costly_10 = pd.read_sql(query, conn)
sql_result_costly_10

In [ ]:
# Compare top 10 costly books (Pandas merge vs SQL join)
pandas_result_costly_10 = merged_df.sort_values(by=['price_inr','title'], ascending=[False,True]).head(10)[['title', 'category_name', 'rating','price_inr']]
pandas_result_costly_10

In [37]:
# Final check: verify SQL and Pandas results match
pd.testing.assert_frame_equal(sql_result_costly_10.reset_index(drop=True), pandas_result_costly_10.reset_index(drop=True), check_dtype=False)
print("SQL and Pandas results match perfectly!")

SQL and Pandas results match perfectly!


In [33]:
# Cleanup: Close the connection
conn.close()